# Transformer Encoder and Decoder

## Learning Objectives
- Understand the roles of encoder and decoder in Transformer models.
- Learn how these components interact for sequence-to-sequence tasks.

## Introduction
Transformer models use an encoder to process input sequences and a decoder to generate outputs, commonly applied in machine translation and text generation.

## Core Concepts

- Encoder: The job of the encoder is to "understand" the input sequence. It's made of a stack of identical layers (e.g., 6 layers). Each layer has two main parts: a multi-head self-attention mechanism to create contextual representations of the input, followed by a simple feed-forward neural network to process these representations further. The output of the final encoder layer is a rich set of numerical representations for the entire input sequence, ready to be used by the decoder.

- Decoder: The job of the decoder is to generate the output sequence, one token at a time, using the information from the encoder. It is also a stack of identical layers, but each layer has three parts:

    - A masked multi-head self-attention layer to look at the previously generated output tokens.
    - An encoder-decoder attention (or cross-attention) layer that looks at the output of the encoder, allowing the decoder to focus on relevant parts of the original input sequence.
    - A feed-forward neural network.

- Masked Self-Attention: This is a critical detail for the decoder. When generating a sequence, the model should only be able to use the tokens it has already produced. It can't be allowed to "cheat" by looking at the next word in the sequence it is trying to predict. The mask is applied to the self-attention scores inside the decoder, setting the scores for all future tokens to negative infinity. After the softmax function, this forces the attention weights for future tokens to be zero, ensuring the model is causal and only attends to the past.

## Example
Code for encoder-decoder interaction.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- 1. Data Preparation ---
VOCAB_SIZE = 100
MAX_SEQ_LENGTH = 10
NUM_SAMPLES = 3000
EMBED_DIM = 128 # Embedding dimension

input_sequences = np.random.randint(1, VOCAB_SIZE, size=(NUM_SAMPLES, MAX_SEQ_LENGTH))
target_sequences = np.fliplr(input_sequences)

encoder_input_data = input_sequences
# Decoder input is the target sequence shifted right (starts with 0)
decoder_input_data = np.insert(target_sequences, 0, 0, axis=1)[:, :-1]
# Decoder target is the original target sequence
decoder_target_data = np.expand_dims(target_sequences, -1)


# --- 2. Building a simpler GRU-based Encoder-Decoder Model ---

# --- ENCODER ---
encoder_inputs = keras.Input(shape=(None,), name="encoder_inputs")
x = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(encoder_inputs)
# We discard `encoder_outputs` and only keep the state.
_, state = layers.GRU(EMBED_DIM, return_state=True)(x)
encoder_state = state

# --- DECODER ---
decoder_inputs = keras.Input(shape=(None,), name="decoder_inputs")
# We reuse the same embedding layer for the decoder
decoder_embedding = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
x = decoder_embedding(decoder_inputs)
decoder_gru = layers.GRU(EMBED_DIM, return_sequences=True, return_state=True)
# The decoder is initialized with the encoder's final state
decoder_outputs, _ = decoder_gru(x, initial_state=encoder_state)
decoder_dense = layers.Dense(VOCAB_SIZE, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# --- Create the Model ---
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

# --- 3. Training ---
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

print("\n--- Starting Training ---")
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=10,
    validation_split=0.2,
    verbose=2,
)
print("--- Training Finished ---")


# --- 4. Inference (Sampling) ---
# For inference, we need to build the models separately to feed the state back in a loop

# --- Define Inference Models ---
encoder_model = keras.Model(encoder_inputs, encoder_state)

decoder_state_input = keras.Input(shape=(EMBED_DIM,))
x_dec = decoder_embedding(decoder_inputs)
decoder_outputs, state_out = decoder_gru(x_dec, initial_state=decoder_state_input)
decoder_states = state_out
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs, decoder_state_input], [decoder_outputs, decoder_states]
)

def reverse_sequence(input_seq):
    # Encode the input to get the initial state for the decoder
    state_value = encoder_model.predict(input_seq, verbose=0)
    
    # Start the decoding with a single "start" token (0)
    target_seq = np.zeros((1, 1))
    
    reversed_sequence = []
    for _ in range(MAX_SEQ_LENGTH):
        output_tokens, h = decoder_model.predict([target_seq, state_value], verbose=0)
        
        # Get the most likely next token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        
        # Stop if we generate the padding token
        if sampled_token_index == 0:
            break
            
        reversed_sequence.append(int(sampled_token_index))
        
        # Update the input for the next time step
        target_seq = np.array([[sampled_token_index]])
        state_value = h

    return reversed_sequence

print("\n--- Performing Inference ---")
test_input = input_sequences[0:1]
reversed_output = reverse_sequence(test_input)
print("Input Sequence:", test_input[0])
print("Predicted Reversed Sequence:", np.array(reversed_output))
print("Actual Reversed Sequence:   ", target_sequences[0])


--- Starting Training ---
Epoch 1/10
38/38 - 5s - 127ms/step - accuracy: 0.0122 - loss: 4.6038 - val_accuracy: 0.0110 - val_loss: 4.6021
Epoch 2/10
38/38 - 1s - 33ms/step - accuracy: 0.0137 - loss: 4.5948 - val_accuracy: 0.0120 - val_loss: 4.5993
Epoch 3/10
38/38 - 1s - 29ms/step - accuracy: 0.0150 - loss: 4.5861 - val_accuracy: 0.0092 - val_loss: 4.5975
Epoch 4/10
38/38 - 1s - 29ms/step - accuracy: 0.0177 - loss: 4.5753 - val_accuracy: 0.0098 - val_loss: 4.5974
Epoch 5/10
38/38 - 1s - 28ms/step - accuracy: 0.0214 - loss: 4.5596 - val_accuracy: 0.0085 - val_loss: 4.6036
Epoch 6/10
38/38 - 1s - 29ms/step - accuracy: 0.0231 - loss: 4.5401 - val_accuracy: 0.0108 - val_loss: 4.6162
Epoch 7/10
38/38 - 1s - 29ms/step - accuracy: 0.0241 - loss: 4.5242 - val_accuracy: 0.0095 - val_loss: 4.6332
Epoch 8/10
38/38 - 1s - 29ms/step - accuracy: 0.0273 - loss: 4.5039 - val_accuracy: 0.0095 - val_loss: 4.6396
Epoch 9/10
38/38 - 1s - 28ms/step - accuracy: 0.0291 - loss: 4.4854 - val_accuracy: 0.0100 -

## Exercise
Explain why masking is important in the decoder self-attention layer.

In [ ]:
# Your notes here

## Summary
- Encoders represent the input context, decoders generate output sequence.
- Masking ensures causal sequential generation in decoding.


## Further Reading
- [Tensor2Tensor Transformer](https://github.com/tensorflow/tensor2tensor/blob/master/tensor2tensor/models/transformer.py)
- [The Annotated Transformer](http://nlp.seas.harvard.edu/2018/04/03/attention.html#the-transformer)
